In [4]:
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

try:
    import lightgbm as lgb
    HAVE_LGB = True
except ImportError:
    HAVE_LGB = False


TRAIN_PATH = r"C:\Users\User\Downloads\Telegram Desktop\train (3).csv"
TEST_PATH = r"C:\Users\User\Downloads\Telegram Desktop\test.csv"
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", ".")

N_FOLDS = int(os.environ.get("N_FOLDS", 5))          
SEEDS = [42, 777, 123, 456, 999]                     
ETA = 0.05
MAX_ROUNDS = int(os.environ.get("MAX_ROUNDS", 3000))
EARLY_STOP = int(os.environ.get("EARLY_STOP", 100))
PRIOR_TE = 200
SEED_CV = 2026
RUN_ADVERSARIAL = os.environ.get("RUN_ADVERSARIAL", "0") == "1"  
RUN_TIME_SPLIT_VALIDATION = os.environ.get("RUN_TIME_SPLIT_VALIDATION", "0") == "1"


USE_DATE_FEATURES = os.environ.get("USE_DATE_FEATURES", "0") == "1"


INCOME_PROXY_FALLBACK_MODE = os.environ.get("INCOME_PROXY_FALLBACK_MODE", "proxy_median")

PARAMS_COMMON = {
    "eta": ETA,
    "max_depth": 7,
    "min_child_weight": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "lambda": 2,
    "tree_method": "hist",
}

LGB_PARAMS = {
    "objective": "regression_l1",
    "metric": "l1",                 
    "learning_rate": ETA,
    "num_leaves": 150,
    "min_data_in_leaf": 40,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 2,
    "verbosity": -1,
}


TRUE_CAT_COLS = [
    "gender", "adminarea", "city_smart_name",
    "dp_ewb_last_employment_position",
    "dp_address_unique_regions", "addrref",
]

NA_STRINGS = ["", "NA", "NaN", "None", "null", "NULL", "nan"]


SEMI_CAT_COLS = ["incomeValueCategory"] 




def wmae(y: np.ndarray, p: np.ndarray, w: np.ndarray) -> float:
    return float(np.sum(w * np.abs(y - p)) / np.sum(w))


def w_median(y: np.ndarray, w: np.ndarray) -> float:
    order = np.argsort(y)
    y_sorted, w_sorted = y[order], w[order]
    cum = np.cumsum(w_sorted) / np.sum(w_sorted)
    idx = int(np.searchsorted(cum, 0.5, side="left"))
    idx = min(idx, len(y_sorted) - 1)
    return float(y_sorted[idx])


def to_numeric_ru(s: pd.Series) -> pd.Series:

    return pd.to_numeric(s.astype("string").str.replace(",", ".", regex=False),
                          errors="coerce").astype("float64")


def read_raw(path: str) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", dtype=str, keep_default_na=False,
                        na_values=NA_STRINGS)


def safe_div(a, b) -> np.ndarray:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = np.where(np.isnan(a) | np.isnan(b) | (b == 0), np.nan, a / b)
    return out


def has_cols(df: pd.DataFrame, cols) -> bool:
    return all(c in df.columns for c in cols)



_POSITION_PATTERNS = {
    "pos_head": r"директор|руковод|начальник|заведующ|управляющ|президент",
    "pos_manager": r"менеджер|администратор",
    "pos_engineer": r"инженер|техник|программист|разработ|\bit\b",
    "pos_spec": r"специалист|эксперт|аналитик|консультант",
    "pos_driver": r"водител|машинист|курьер|экспедитор",
    "pos_sales": r"продавец|кассир|торгов",
    "pos_teacher": r"учител|преподават|воспитат|педагог",
    "pos_med": r"врач|медицин|медсестр|фельдшер",
    "pos_worker": r"рабоч|слесар|монтаж|сварщ|электрик|оператор|грузчик|уборщ|охран|повар",
    "pos_finance": r"бухгалтер|экономист|финанс|юрист",
}


def add_position_flags(df: pd.DataFrame) -> pd.DataFrame:
    col = "dp_ewb_last_employment_position"
    if col not in df.columns:
        return df

    s = df[col].fillna("").astype(str).str.lower()
    for name, pat in _POSITION_PATTERNS.items():
        df[name] = s.str.contains(pat, regex=True, na=False).astype(float)
    df["position_nchar"] = s.str.len().astype(float)
    return df



def compute_income_proxy_raw(df: pd.DataFrame) -> np.ndarray:
    proxy = np.full(len(df), np.nan)

    def step(proxy, col, f=lambda x: x):
        if col not in df.columns:
            return proxy
        v = df[col].to_numpy(dtype=float)
        mask = np.isnan(proxy) & ~np.isnan(v)
        return np.where(mask, f(v), proxy)


    proxy = step(proxy, "salary_6to12m_avg", lambda v: v * 0.957)

    proxy = step(proxy, "first_salary_income", lambda v: v * 0.477)
 
    proxy = step(proxy, "dp_payoutincomedata_payout_avg_6_month", lambda v: v * 0.883)

    proxy = step(proxy, "dp_ils_paymentssum_avg_12m", lambda v: v * 0.409)

    proxy = step(proxy, "turn_cur_db_sum_v2", lambda v: v / 12 * 0.883)

    proxy = step(proxy, "avg_6m_all", lambda v: v * 1.848)
    return proxy


def get_income_proxy(df: pd.DataFrame, fallback: float) -> np.ndarray:
    proxy = compute_income_proxy_raw(df)
    return np.where(np.isnan(proxy), fallback, proxy)



def add_date_features(train_feat, test_feat, train_dt, test_dt):
    min_date = min(train_dt.min(), test_dt.min())

    train_feat["dt_year"] = train_dt.dt.year.astype(float)
    train_feat["dt_month"] = train_dt.dt.month.astype(float)
    train_feat["dt_month_sin"] = np.sin(2 * np.pi * train_dt.dt.month / 12)
    train_feat["dt_month_cos"] = np.cos(2 * np.pi * train_dt.dt.month / 12)
    train_feat["dt_days_since_min"] = (train_dt - min_date).dt.days.astype(float)

    test_feat["dt_year"] = test_dt.dt.year.astype(float)
    test_feat["dt_month"] = test_dt.dt.month.astype(float)
    test_feat["dt_month_sin"] = np.sin(2 * np.pi * test_dt.dt.month / 12)
    test_feat["dt_month_cos"] = np.cos(2 * np.pi * test_dt.dt.month / 12)
    test_feat["dt_days_since_min"] = (test_dt - min_date).dt.days.astype(float)

    return train_feat, test_feat



def make_stratified_folds(y_log: np.ndarray, n_folds: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    qs = np.unique(np.quantile(y_log, np.arange(11) / 10))  # type=7 == numpy 'linear'
    dec = np.asarray(pd.cut(y_log, bins=qs, include_lowest=True, labels=False))
    n = len(y_log)
    fold_id = np.zeros(n, dtype=int)
    for d in np.unique(dec):
        idx = np.where(dec == d)[0]
        labels = np.resize(np.arange(1, n_folds + 1), len(idx))
        rng.shuffle(labels)
        fold_id[idx] = labels
    return fold_id



def add_cat_encodings(train_feat, test_feat, cat_cols, y_log, w, fold_id, prior):
    global_m = float(np.sum(w * y_log) / np.sum(w))
    n = len(train_feat)
    cols_present = [c for c in cat_cols if c in train_feat.columns]

    for col in cols_present:
        tr_v = train_feat[col].astype("string")
        tr_v = tr_v.fillna("__NA__").to_numpy(dtype=object)
        te_v = test_feat[col].astype("string")
        te_v = te_v.fillna("__NA__").to_numpy(dtype=object)

        te_col = np.full(n, global_m, dtype=float)
        for f in np.unique(fold_id):
            in_i = fold_id != f
            out_i = ~in_i
            grp = pd.DataFrame({"v": tr_v[in_i], "wy": w[in_i] * y_log[in_i],
                                 "w": w[in_i]}).groupby("v", sort=False).sum()
            enc = (grp["wy"] + prior * global_m) / (grp["w"] + prior)
            mapped = pd.Series(tr_v[out_i]).map(enc).fillna(global_m)
            te_col[out_i] = mapped.to_numpy(dtype=float)

        grp_f = pd.DataFrame({"v": tr_v, "wy": w * y_log,
                               "w": w}).groupby("v", sort=False).sum()
        enc_f = (grp_f["wy"] + prior * global_m) / (grp_f["w"] + prior)
        te_te = pd.Series(te_v).map(enc_f).fillna(global_m).to_numpy(dtype=float)

        cnt = pd.Series(tr_v).value_counts()
        cnt_tr = pd.Series(tr_v).map(cnt).fillna(0).to_numpy(dtype=float)
        cnt_te = pd.Series(te_v).map(cnt).fillna(0).to_numpy(dtype=float)

        lv = sorted(pd.unique(tr_v).tolist())
        codes_tr = pd.Categorical(tr_v, categories=lv).codes
        codes_te = pd.Categorical(te_v, categories=lv).codes
        le_tr = np.where(codes_tr == -1, np.nan, codes_tr + 1).astype(float)
        le_te = np.where(codes_te == -1, np.nan, codes_te + 1).astype(float)

        train_feat[f"te_{col}"] = te_col
        test_feat[f"te_{col}"] = te_te
        train_feat[f"cnt_{col}"] = cnt_tr
        test_feat[f"cnt_{col}"] = cnt_te
        train_feat[f"le_{col}"] = le_tr
        test_feat[f"le_{col}"] = le_te

        train_feat = train_feat.drop(columns=[col])
        test_feat = test_feat.drop(columns=[col])

    return train_feat, test_feat



def xgb_importance_df(model: xgb.Booster) -> pd.DataFrame:
    total_gain = model.get_score(importance_type="total_gain")
    total_cover = model.get_score(importance_type="total_cover")
    freq = model.get_score(importance_type="weight")
    feats = list(total_gain.keys())
    if not feats:
        return pd.DataFrame(columns=["Feature", "Gain", "Cover", "Frequency"])
    gain_sum = sum(total_gain.values())
    cover_sum = sum(total_cover.values()) or 1.0
    freq_sum = sum(freq.values()) or 1.0
    df = pd.DataFrame({
        "Feature": feats,
        "Gain": [total_gain[ft] / gain_sum for ft in feats],
        "Cover": [total_cover.get(ft, 0.0) / cover_sum for ft in feats],
        "Frequency": [freq.get(ft, 0.0) / freq_sum for ft in feats],
    })
    return df.sort_values("Gain", ascending=False).reset_index(drop=True)


def wmae_eval_raw(preds, dtrain):
    y, w = dtrain.get_label(), dtrain.get_weight()
    return "wmae", float(np.sum(w * np.abs(y - preds)) / np.sum(w))


def wmae_eval_log(preds, dtrain):
    y, w = dtrain.get_label(), dtrain.get_weight()
    return "wmae", float(np.sum(w * np.abs(np.expm1(y) - np.expm1(preds))) / np.sum(w))


def kind_spec(kind, y_log, train_target):
    specs = {
        "xgb_log": dict(lab=y_log, obj="reg:squarederror", log=True, fe=wmae_eval_log),
        "xgb_mae": dict(lab=train_target, obj="reg:absoluteerror", log=False, fe=wmae_eval_raw),
        "xgb_logmae": dict(lab=y_log, obj="reg:absoluteerror", log=True, fe=wmae_eval_log),
    }
    return specs[kind]


def run_cv(kind, X_train, X_test, d_test_xgb, train_target, train_w, y_log,
           fold_id, n_folds, pred_cols):
    n_tr = X_train.shape[0]
    oof = np.full(n_tr, np.nan)
    best_iters = []
    test_bag = np.zeros(X_test.shape[0])

    for f in range(1, n_folds + 1):
        tr_mask = fold_id != f
        va_mask = ~tr_mask

        if kind == "lgb_l1":
            dtr = lgb.Dataset(X_train[tr_mask], label=train_target[tr_mask],
                               weight=train_w[tr_mask], feature_name=pred_cols)
            dva = lgb.Dataset(X_train[va_mask], label=train_target[va_mask],
                               weight=train_w[va_mask], reference=dtr,
                               feature_name=pred_cols)
            params = dict(LGB_PARAMS, seed=100 + f)
            m = lgb.train(params, dtr, num_boost_round=MAX_ROUNDS, valid_sets=[dva],
                           callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                                      lgb.log_evaluation(0)])
            b = m.best_iteration if m.best_iteration else MAX_ROUNDS
            p = m.predict(X_train[va_mask], num_iteration=b)
            pt = m.predict(X_test, num_iteration=b)
        else:
            sp = kind_spec(kind, y_log, train_target)
            dtr = xgb.DMatrix(X_train[tr_mask], label=sp["lab"][tr_mask],
                               weight=train_w[tr_mask], missing=np.nan,
                               feature_names=pred_cols)
            dva = xgb.DMatrix(X_train[va_mask], label=sp["lab"][va_mask],
                               weight=train_w[va_mask], missing=np.nan,
                               feature_names=pred_cols)
            params = dict(PARAMS_COMMON, objective=sp["obj"], seed=100 + f)
            m = xgb.train(params, dtr, num_boost_round=MAX_ROUNDS,
                           evals=[(dva, "valid")],
                           custom_metric=sp["fe"], maximize=False,
                           early_stopping_rounds=EARLY_STOP, verbose_eval=False)
            bi = getattr(m, "best_iteration", None)
            b = MAX_ROUNDS if bi is None else min(int(bi) + 1, MAX_ROUNDS)
            p = m.predict(dva, iteration_range=(0, b))
            pt = m.predict(d_test_xgb, iteration_range=(0, b))
            if sp["log"]:
                p = np.expm1(p)
                pt = np.expm1(pt)

        best_iters.append(b)
        oof[va_mask] = p
        test_bag += pt / n_folds
        fold_wmae = wmae(train_target[va_mask], p, train_w[va_mask])
        print(f"  [{kind}] fold {f}: best_iter={b}, fold WMAE={fold_wmae:.1f}")

    return dict(oof=oof, best_iters=np.array(best_iters), test_bag=test_bag)



def full_predict(kind, nrounds_k, X_train, X_test, d_test_xgb, train_target,
                  train_w, y_log, seeds, pred_cols):
    acc = np.zeros(X_test.shape[0])
    saved_model = None
    for s in seeds:
        if kind == "lgb_l1":
            dfull = lgb.Dataset(X_train, label=train_target, weight=train_w,
                                 feature_name=pred_cols)
            params = dict(LGB_PARAMS, seed=s, bagging_seed=s,
                           feature_fraction_seed=s, data_random_seed=s)
            m = lgb.train(params, dfull, num_boost_round=nrounds_k)
            p = m.predict(X_test, num_iteration=nrounds_k)
        else:
            sp = kind_spec(kind, y_log, train_target)
            d = xgb.DMatrix(X_train, label=sp["lab"], weight=train_w,
                             missing=np.nan, feature_names=pred_cols)
            params = dict(PARAMS_COMMON, objective=sp["obj"], seed=s)
            m = xgb.train(params, d, num_boost_round=nrounds_k, verbose_eval=False)
            p = m.predict(d_test_xgb)
            if sp["log"]:
                p = np.expm1(p)
            if kind == "xgb_log" and s == seeds[0]:
                saved_model = m
        acc += p / len(seeds)
    return acc, saved_model



def main():
    t0 = time.time()
    warnings.filterwarnings("ignore")


    train = read_raw(TRAIN_PATH)
    test = read_raw(TEST_PATH)


    numeric_like_cols = [c for c in train.columns
                          if c not in ["id", "dt"] + TRUE_CAT_COLS]
    for col in numeric_like_cols:
        train[col] = to_numeric_ru(train[col])
        if col in test.columns:
            test[col] = to_numeric_ru(test[col])

    train_id = train["id"].to_numpy()
    test_id = test["id"].to_numpy()
    train_target = train["target"].to_numpy(dtype=float)
    train_w = train["w"].to_numpy(dtype=float)


    train_dt = pd.to_datetime(train["dt"], errors="coerce")
    test_dt = pd.to_datetime(test["dt"], errors="coerce")
    train_feat = train.drop(columns=[c for c in ["id", "dt", "target", "w"]
                                      if c in train.columns])
    test_feat = test.drop(columns=[c for c in ["id", "dt"] if c in test.columns])


    valid_rows = (~np.isnan(train_target)) & (~np.isnan(train_w)) & \
                 np.isfinite(train_target) & np.isfinite(train_w)
    if not valid_rows.all():
        print(f"Удалено строк с невалидными target/w: {int((~valid_rows).sum())}")
        train_feat = train_feat.loc[valid_rows].reset_index(drop=True)
        train_target = train_target[valid_rows]
        train_w = train_w[valid_rows]
        train_id = train_id[valid_rows]
        train_dt = train_dt[valid_rows].reset_index(drop=True)

    n_tr = len(train_feat)
    y_log = np.log1p(train_target)

    print("Baseline (константа = взвеш. медиана), WMAE:",
          round(wmae(train_target, w_median(train_target, train_w), train_w), 1))

  
    num_cols0 = train_feat.select_dtypes(include=[np.number]).columns.tolist()
    for col in num_cols0:
        x = train_feat[col].to_numpy(dtype=float)
        if np.sum(~np.isnan(x)) < 20:
            continue
        q_lo, q_hi = np.nanquantile(x, [0.005, 0.995], method="linear")
        if not (np.isfinite(q_lo) and np.isfinite(q_hi)) or q_lo == q_hi:
            continue
        train_feat[col] = train_feat[col].clip(lower=q_lo, upper=q_hi)
        if col in test_feat.columns:
            test_feat[col] = test_feat[col].clip(lower=q_lo, upper=q_hi)

   
    miss_perc = train_feat[num_cols0].isna().mean() * 100
    na_flag_cols = miss_perc[miss_perc > 20].index.tolist()
    for col in na_flag_cols:
        train_feat[f"{col}_isna"] = train_feat[col].isna().astype(float)
        if col in test_feat.columns:
            test_feat[f"{col}_isna"] = test_feat[col].isna().astype(float)

    test_cols_present = [c for c in num_cols0 if c in test_feat.columns]
    train_feat["n_missing"] = train_feat[num_cols0].isna().sum(axis=1)
    test_feat["n_missing"] = test_feat[test_cols_present].isna().sum(axis=1)
    train_feat["completeness"] = train_feat[num_cols0].notna().mean(axis=1)
    test_feat["completeness"] = test_feat[test_cols_present].notna().mean(axis=1)

    grp_prefixes = ["dp_", "hdb_", "turn_", "avg_", "bki_", "transaction",
                     "by_category", "vert_"]
    for p in grp_prefixes:
        cols = [c for c in num_cols0 if c.startswith(p)]
        if len(cols) >= 3:
            nm = f"nfill_{re.sub(r'[^a-z]+$', '', p)}"
            cols_te = [c for c in cols if c in test_feat.columns]
            train_feat[nm] = train_feat[cols].notna().sum(axis=1)
            test_feat[nm] = test_feat[cols_te].notna().sum(axis=1)


    train_feat = add_position_flags(train_feat)
    test_feat = add_position_flags(test_feat)


    if USE_DATE_FEATURES:
        train_feat, test_feat = add_date_features(train_feat, test_feat, train_dt, test_dt)
    else:
        print("USE_DATE_FEATURES=0 — фичи из dt не добавляются "
              "(train/test не пересекаются по месяцам).")


    if has_cols(train_feat, ["hdb_outstand_sum", "hdb_bki_total_max_limit"]):
        train_feat["credit_util"] = safe_div(train_feat["hdb_outstand_sum"],
                                              train_feat["hdb_bki_total_max_limit"])
        test_feat["credit_util"] = safe_div(test_feat["hdb_outstand_sum"],
                                             test_feat["hdb_bki_total_max_limit"])
    if has_cols(train_feat, ["avg_cur_db_turn", "turn_cur_db_avg_v2"]):
        train_feat["trend_db"] = safe_div(train_feat["avg_cur_db_turn"],
                                           train_feat["turn_cur_db_avg_v2"])
        test_feat["trend_db"] = safe_div(test_feat["avg_cur_db_turn"],
                                          test_feat["turn_cur_db_avg_v2"])
    if has_cols(train_feat, ["turn_cur_db_sum_v2", "avg_6m_all"]):
        train_feat["savings_rate"] = safe_div(
            train_feat["turn_cur_db_sum_v2"] - train_feat["avg_6m_all"] * 12,
            train_feat["turn_cur_db_sum_v2"])
        test_feat["savings_rate"] = safe_div(
            test_feat["turn_cur_db_sum_v2"] - test_feat["avg_6m_all"] * 12,
            test_feat["turn_cur_db_sum_v2"])


    global_target_med = float(np.median(train_target))
    train_income_proxy_raw = compute_income_proxy_raw(train_feat)
    proxy_median = float(np.nanmedian(train_income_proxy_raw))
    if INCOME_PROXY_FALLBACK_MODE == "target_median":
        income_proxy_fallback = global_target_med
    else:
        income_proxy_fallback = proxy_median if np.isfinite(proxy_median) else global_target_med
    print(f"income_proxy fallback ({INCOME_PROXY_FALLBACK_MODE}): "
          f"{income_proxy_fallback:,.0f} "
          f"[proxy_median={proxy_median:,.0f}, target_median={global_target_med:,.0f}]")
    train_feat["income_proxy"] = np.where(np.isnan(train_income_proxy_raw),
                                           income_proxy_fallback,
                                           train_income_proxy_raw)
    test_feat["income_proxy"] = get_income_proxy(test_feat, income_proxy_fallback)

    if has_cols(train_feat, ["hdb_outstand_sum"]):
        train_feat["debt_to_income"] = safe_div(train_feat["hdb_outstand_sum"],
                                                 train_feat["income_proxy"])
        test_feat["debt_to_income"] = safe_div(test_feat["hdb_outstand_sum"],
                                                test_feat["income_proxy"])
    if has_cols(train_feat, ["hdb_bki_total_max_limit"]):
        train_feat["limit_to_income"] = safe_div(train_feat["hdb_bki_total_max_limit"],
                                                  train_feat["income_proxy"])
        test_feat["limit_to_income"] = safe_div(test_feat["hdb_bki_total_max_limit"],
                                                 test_feat["income_proxy"])
    if has_cols(train_feat, ["per_capita_income_rur_amt"]):
        train_feat["income_to_region"] = safe_div(train_feat["income_proxy"],
                                                    train_feat["per_capita_income_rur_amt"])
        test_feat["income_to_region"] = safe_div(test_feat["income_proxy"],
                                                   test_feat["per_capita_income_rur_amt"])
        if has_cols(train_feat, ["turn_cur_cr_avg_act_v2"]):
            train_feat["turn_to_region"] = safe_div(train_feat["turn_cur_cr_avg_act_v2"],
                                                      train_feat["per_capita_income_rur_amt"])
            test_feat["turn_to_region"] = safe_div(test_feat["turn_cur_cr_avg_act_v2"],
                                                     test_feat["per_capita_income_rur_amt"])


    q_inc = np.unique(np.nanquantile(train_feat["income_proxy"].to_numpy(dtype=float),
                                      [0.2, 0.4, 0.6, 0.8], method="linear"))
    seg_bins = np.concatenate(([-np.inf], q_inc, [np.inf]))

    def seg_num(x):
        return np.asarray(pd.cut(x, bins=seg_bins, labels=False), dtype=float) + 1

    train_feat["segment_num"] = seg_num(train_feat["income_proxy"])
    test_feat["segment_num"] = seg_num(test_feat["income_proxy"])


    fold_id = make_stratified_folds(y_log, N_FOLDS, SEED_CV)

 
    train_feat, test_feat = add_cat_encodings(
        train_feat, test_feat, TRUE_CAT_COLS, y_log, train_w, fold_id, PRIOR_TE)


    semi_present = [c for c in SEMI_CAT_COLS if c in train_feat.columns]
    if semi_present:

        saved_tr = {c: train_feat[c].copy() for c in semi_present}
        saved_te = {c: test_feat[c].copy() if c in test_feat.columns
                    else pd.Series(np.nan, index=range(len(test_feat))) for c in semi_present}
        train_feat, test_feat = add_cat_encodings(
            train_feat, test_feat, semi_present, y_log, train_w, fold_id, PRIOR_TE)
  
        for c in semi_present:
            train_feat[c] = saved_tr[c]
            test_feat[c] = saved_te[c]


    pred_cols = train_feat.select_dtypes(include=[np.number]).columns.tolist()
    for c0 in pred_cols:
        if c0 not in test_feat.columns:
            test_feat[c0] = np.nan
    X_train = train_feat[pred_cols].to_numpy(dtype=float)
    X_test = test_feat[pred_cols].to_numpy(dtype=float)
    comp_tr = train_feat["completeness"].to_numpy(dtype=float)
    comp_te = test_feat["completeness"].to_numpy(dtype=float)


    d_test_xgb = xgb.DMatrix(X_test, missing=np.nan, feature_names=pred_cols)

  
    if RUN_ADVERSARIAL:
        print("\nAdversarial validation (train vs test)")
        X_adv = np.vstack([X_train, X_test])
        y_adv = np.concatenate([np.zeros(X_train.shape[0]), np.ones(X_test.shape[0])])
        rng_adv = np.random.default_rng(1)
        idx = rng_adv.permutation(X_adv.shape[0])
        n_va = round(0.25 * len(idx))
        va_i, tr_i = idx[:n_va], idx[n_va:]
        d_atr = xgb.DMatrix(X_adv[tr_i], label=y_adv[tr_i], missing=np.nan,
                             feature_names=pred_cols)
        d_ava = xgb.DMatrix(X_adv[va_i], label=y_adv[va_i], missing=np.nan,
                             feature_names=pred_cols)
        m_adv = xgb.train(
            params={"objective": "binary:logistic", "eval_metric": "auc",
                    "eta": 0.1, "max_depth": 5, "tree_method": "hist"},
            dtrain=d_atr, num_boost_round=300,
            evals=[(d_ava, "valid")],
            early_stopping_rounds=30, verbose_eval=False)
        p_adv = m_adv.predict(d_ava)
        auc = roc_auc_score(y_adv[va_i], p_adv)
        print(f"AUC train-vs-test: {auc:.3f} "
              "(~0.5 = сдвига нет; >0.6 — проверьте топ-фичи ниже)")
        print(xgb_importance_df(m_adv).head(15))

    if RUN_TIME_SPLIT_VALIDATION:
        print("\n=== Time-split validation (train на ранних датах / valid на поздних) ===")
        order = np.argsort(train_dt.to_numpy())
        n_va_ts = int(round(0.25 * len(order)))
        va_i, tr_i = order[-n_va_ts:], order[:-n_va_ts]
        sp_ts = kind_spec("xgb_log", y_log, train_target)
        d_tr_ts = xgb.DMatrix(X_train[tr_i], label=sp_ts["lab"][tr_i],
                               weight=train_w[tr_i], missing=np.nan,
                               feature_names=pred_cols)
        d_va_ts = xgb.DMatrix(X_train[va_i], label=sp_ts["lab"][va_i],
                               weight=train_w[va_i], missing=np.nan,
                               feature_names=pred_cols)
        m_ts = xgb.train(dict(PARAMS_COMMON, objective=sp_ts["obj"], seed=42),
                          d_tr_ts, num_boost_round=MAX_ROUNDS,
                          evals=[(d_va_ts, "valid")],
                          custom_metric=sp_ts["fe"], maximize=False,
                          early_stopping_rounds=EARLY_STOP, verbose_eval=False)
        bi_ts = getattr(m_ts, "best_iteration", None)
        b_ts = MAX_ROUNDS if bi_ts is None else min(int(bi_ts) + 1, MAX_ROUNDS)
        p_ts = np.expm1(m_ts.predict(d_va_ts, iteration_range=(0, b_ts)))
        ts_wmae = wmae(train_target[va_i], p_ts, train_w[va_i])


 
    KINDS = ["xgb_log", "xgb_mae"] + (["lgb_l1"] if HAVE_LGB else ["xgb_logmae"])
    extra = "" if HAVE_LGB else " (lightgbm не найден — заменён на xgb MAE-на-log)"
    print(f"\nМодели в ансамбле: {', '.join(KINDS)}{extra}")

    results = {}
    for k in KINDS:
        print(f"\nCV: {k}")
        results[k] = run_cv(k, X_train, X_test, d_test_xgb, train_target, train_w,
                             y_log, fold_id, N_FOLDS, pred_cols)
        print(f"OOF WMAE ({k}):",
              round(wmae(train_target, results[k]["oof"], train_w), 1))


    oof_mat = np.column_stack([results[k]["oof"] for k in KINDS])
    K = oof_mat.shape[1]
    rng_w = np.random.default_rng(7)
    dirichlet_draws = rng_w.exponential(1.0, size=(4000, K))
    dirichlet_draws /= dirichlet_draws.sum(axis=1, keepdims=True)
    W = np.vstack([np.eye(K), np.full((1, K), 1.0 / K), dirichlet_draws])

    sc = np.empty(W.shape[0])
    for i in range(0, W.shape[0], 200):
        j = slice(i, min(i + 200, W.shape[0]))
        P = oof_mat @ W[j].T
        sc[j] = (train_w[:, None] * np.abs(P - train_target[:, None])).sum(axis=0) \
            / train_w.sum()

    best_idx = int(np.argmin(sc))
    w_best = W[best_idx]
    oof_blend = oof_mat @ w_best
    print(f"\nВеса ансамбля ({'/'.join(KINDS)}): "
          f"{' / '.join(f'{v:.3f}' for v in w_best)} -> OOF WMAE = {sc[best_idx]:.1f}")

 
    cals = np.round(np.arange(0.85, 1.15 + 1e-9, 0.005), 10)
    sc_c = np.array([wmae(train_target, cc * oof_blend, train_w) for cc in cals])
    c_global = float(cals[np.argmin(sc_c)])


    qs_comp = np.unique(np.quantile(comp_tr, [0.25, 0.5, 0.75], method="linear"))

    def grp_of(x):
        return np.searchsorted(qs_comp, x, side="right") + 1

    g_tr, g_te = grp_of(comp_tr), grp_of(comp_te)
    n_grp = int(g_tr.max())
    c_comp = np.full(n_grp + 1, c_global, dtype=float)
    for g in range(1, n_grp + 1):
        ii = np.where(g_tr == g)[0]
        if len(ii) < 3000:
            continue
        sc_g = np.array([wmae(train_target[ii], cc * oof_blend[ii], train_w[ii])
                          for cc in cals])
        c_comp[g] = cals[np.argmin(sc_g)]

    oof_step1 = oof_blend * c_comp[g_tr]

    ivc_tr = train_feat["incomeValueCategory"].to_numpy(dtype=float) \
        if "incomeValueCategory" in train_feat.columns else \
        np.full(n_tr, np.nan)
    ivc_te = test_feat["incomeValueCategory"].to_numpy(dtype=float) \
        if "incomeValueCategory" in test_feat.columns else \
        np.full(X_test.shape[0], np.nan)

    ivc_vals = np.unique(ivc_tr[~np.isnan(ivc_tr)])
    c_ivc = {} 
    for iv in ivc_vals:
        ii = np.where(ivc_tr == iv)[0]
        if len(ii) < 3000:
            continue
        sc_iv = np.array([wmae(train_target[ii], cc * oof_step1[ii], train_w[ii])
                           for cc in cals])
        c_ivc[iv] = cals[np.argmin(sc_iv)]

    def apply_ivc_cal(pred, ivc_arr):
        out = pred.copy()
        for iv, cv in c_ivc.items():
            mask = ivc_arr == iv
            out[mask] *= cv
        return out

    oof_final = apply_ivc_cal(oof_step1, ivc_tr)

    print("Глобальный множитель:", c_global)
    print("По completeness:", " / ".join(f"{v:.3f}" for v in c_comp[1:]))
    print("По IVC:", {int(k): round(v, 3) for k, v in sorted(c_ivc.items())})
    print("OOF WMAE (completeness × IVC калибровка):",
          round(wmae(train_target, oof_final, train_w), 1))
    print("  vs только completeness:",
          round(wmae(train_target, oof_step1, train_w), 1))


    final_log_model = None
    test_mat = np.zeros((X_test.shape[0], K))
    for k_idx, k in enumerate(KINDS):
        nr = max(200, int(np.ceil(1.05 * np.mean(results[k]["best_iters"]))))
        print(f"Финальный retrain: {k} | nrounds = {nr}")
        full, model_k = full_predict(k, nr, X_train, X_test, d_test_xgb,
                                      train_target, train_w, y_log, SEEDS, pred_cols)
        if model_k is not None:
            final_log_model = model_k
        test_mat[:, k_idx] = 0.5 * results[k]["test_bag"] + 0.5 * full

    test_pred_raw = (test_mat @ w_best) * c_comp[g_te]
    test_pred = apply_ivc_cal(test_pred_raw, ivc_te)
    lo = float(train_target.min())
    hi = float(np.quantile(train_target, 0.999, method="linear"))
    test_pred = np.clip(test_pred, lo, hi)


    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if final_log_model is not None:
        imp = xgb_importance_df(final_log_model)
        print("\nТоп-30 фичей (xgb_log):")
        print(imp.head(30))
        imp.to_csv(os.path.join(OUTPUT_DIR, "feature_importance.csv"),
                    sep=";", decimal=",", index=False)

    submission = pd.DataFrame({"id": test_id, "predict": test_pred})
    submission.to_csv(os.path.join(OUTPUT_DIR, "submission_v2.csv"),
                       sep=";", decimal=",", index=False)

    oof_out = pd.DataFrame({"id": train_id, "target": train_target, "w": train_w,
                             "fold": fold_id})
    for k_idx, k in enumerate(KINDS):
        oof_out[k] = oof_mat[:, k_idx]
    oof_out["oof_final"] = oof_final
    oof_out.to_csv(os.path.join(OUTPUT_DIR, "oof_predictions_v2.csv"),
                    sep=";", decimal=",", index=False)


    print("OOF WMAE финальный:", round(wmae(train_target, oof_final, train_w), 1))



if __name__ == "__main__":
    main()


Baseline (константа = взвеш. медиана), WMAE: 131764.2
USE_DATE_FEATURES=0 — фичи из dt не добавляются (train/test не пересекаются по месяцам).
income_proxy fallback (proxy_median): 45,496 [proxy_median=45,496, target_median=62,754]

Модели в ансамбле: xgb_log, xgb_mae, lgb_l1

CV: xgb_log
  [xgb_log] fold 1: best_iter=379, fold WMAE=62873.5
  [xgb_log] fold 2: best_iter=297, fold WMAE=65062.2
  [xgb_log] fold 3: best_iter=246, fold WMAE=64483.8
  [xgb_log] fold 4: best_iter=278, fold WMAE=65244.3
  [xgb_log] fold 5: best_iter=277, fold WMAE=62464.8
OOF WMAE (xgb_log): 64026.0

CV: xgb_mae
  [xgb_mae] fold 1: best_iter=1037, fold WMAE=60625.9
  [xgb_mae] fold 2: best_iter=444, fold WMAE=63864.9
  [xgb_mae] fold 3: best_iter=325, fold WMAE=63094.1
  [xgb_mae] fold 4: best_iter=716, fold WMAE=63120.0
  [xgb_mae] fold 5: best_iter=315, fold WMAE=61759.2
OOF WMAE (xgb_mae): 62493.6

CV: lgb_l1
  [lgb_l1] fold 1: best_iter=397, fold WMAE=59911.9
  [lgb_l1] fold 2: best_iter=2087, fold WMAE=6